Dataset exported from Zendesk Explore with Real Tickets = Real filter applied. All 4585 rows are valid tickets. No further noise filtering required.

In [4]:
import pandas as pd
df = pd.read_excel('Zendesk_Voice_Tickets_02202026_1125.xlsx', sheet_name='Sheet1')
print(df.shape)
print(df.columns.tolist())


c:\Users\rstwr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


ValueError: Worksheet named 'Sheet1' not found

In [5]:
import pandas as pd
x1 = pd.ExcelFile('Zendesk_Voice_Tickets_02202026_1125.xlsx')
print(x1.sheet_names)

['Zendesk Voice Tickets']


c:\Users\rstwr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [6]:
import pandas as pd
df = pd.read_excel('Zendesk_Voice_Tickets_02202026_1125.xlsx')
print(df.shape)
print(df.columns.tolist())

(4585, 36)
['Ticket ID', 'Ticket created - Timestamp', 'Brand', 'Org', 'Requester', 'Ticket subject', 'Category', 'Category Other Description', 'Resolution', 'Ticket solved - Date', 'Status', 'Ticket priority', 'Submitter name', 'AI Issue Type', 'AI Issue Tag', 'AI Issue Summary', 'Example Session IDs', 'Ticket problem ID', 'Tech Revisit/RMA Required?', 'Drive Thru Down?', 'Linked Jira', 'Jira Status (Automated)', 'Internal vs External Submitter Attribute', 'HITL/PD Submitted', 'Real Tickets', 'Customer Sentiment', 'Sentiment Detail', 'AI Escalation Message', 'Agent intercepted without cause', 'Agent made errors after intercepting', 'Al issue triggered escalation to agent', 'Expected Al escalation to agent', 'Issue caused staff to void/change final POS cart', 'Issue caused staff to intervene', 'Audio Recording Attached', 'Tickets']


In [7]:
df['Category'].value_counts()

Category
Ordering Issue::Order Accuracy                                                   630
Menu::Item Availability/86 Request                                               624
Menu::Menu Update/LTO Request                                                    472
Alert::ENOPOSRESPONSE                                                            379
Alert::LogMeIn ALERT                                                             323
Alert::DT-Prem is Down!                                                          248
Config Change::Voice Off Temporary                                               242
Ordering Issue::DT-Server Issue                                                  164
Menu::Menu Config Issue                                                          156
cat_menu__invalid_item_issue                                                     129
Ordering Issue::POS Issue                                                        123
Alert::High Memory Usage On Site                        

In [8]:
df.groupby('Category')['Drive Thru Down?'].value_counts()

Category                                    Drive Thru Down?
Alert::DT-Prem is Down!                     False               248
Alert::DT-Server Disk Usage                 False                12
Alert::DTBridge Disconnect                  False                 7
Alert::ENOOCBRESPONSE                       False                 2
Alert::ENOPOSRESPONSE                       False               379
                                                               ... 
Vehicle Detection Issue::Low Voltage        False                 8
Vehicle Detection Issue::Voltage Not Found  False                 1
cat_menu__invalid_item_issue                False               129
cat_menu__pricing_issue                     False                 8
                                            False                30
Name: count, Length: 61, dtype: int64

In [9]:
df['Drive Thru Down?'].value_counts()

Drive Thru Down?
False    4525
True       60
Name: count, dtype: int64

In [10]:
df[df['Drive Thru Down?'] == True]['Category'].value_counts()

Category
Audio Issue::No Audio                                             33
Audio Issue::Drive Thru Speaker                                   21
Audio Issue::HITL                                                  2
Audio Issue::Other                                                 2
Ordering Issue::DT-Server Issue                                    1
Vehicle Detection Issue::Customer-side vehicle detection issue     1
Name: count, dtype: int64

In [ ]:
df_real = df[df['Real Tickets'] == True]
print(df_real.shape)

(0, 36)


df_real filters to Real Tickets == True, excluding junk tickets (abandoned calls, misrouted emails, spam). Used as the base dataset for all subsequent analysis, equivalent to the default filter applied in Zendesk Explore dashboards.

In [12]:
df['Real Tickets'].value_counts()

Real Tickets
Real    4585
Name: count, dtype: int64

In [13]:
df['Ticket created - Timestamp'].dtype

<StringDtype(storage='python', na_value=nan)>

In [15]:
df['Ticket created - Timestamp'].head()

0    2026-01-01T03:02:25
1    2026-01-01T04:12:45
2    2026-01-01T04:16:25
3    2026-01-01T05:02:08
4    2026-01-01T05:06:43
Name: Ticket created - Timestamp, dtype: str

In [16]:
df['Ticket created - Timestamp'] = pd.to_datetime(df['Ticket created - Timestamp'])

In [17]:
df['Ticket created - Timestamp'].dtype

dtype('<M8[us]')

In [18]:
df['Month'] = df['Ticket created - Timestamp'].dt.month_name()
df['Month'].value_counts()

Month
January     2789
February    1796
Name: count, dtype: int64

In [19]:
monthly = df['Month'].value_counts()
print(f"Change: {monthly['February'] - monthly['January']}")
print (f"Percentage Change: {((monthly['February'] - monthly['January']) / monthly['January'] * 100).round(1)}%")

Change: -993
Percentage Change: -35.6%


In [20]:
df.groupby(['Month', 'Brand'])['Ticket ID'].count().unstack()

Brand,Carl's Jr,Dairy Queen,Fazoli's,Hardee's,Taco John's,Wienerschnitzel,
Month,,,,,,,
February,517,130,228,124,211,499,87
January,899,118,293,127,446,858,48


In [21]:
brand_monthly = df.groupby(['Month', 'Brand'])['Ticket ID'].count().unstack()
brand_monthly.loc['Change'] = brand_monthly.loc['February'] - brand_monthly.loc['January']
brand_monthly.loc['% Change'] = ((brand_monthly.loc['Change'] / brand_monthly.loc['January']) * 100).round(1)
print(brand_monthly)

Brand     Carl's Jr  Dairy Queen  Fazoli's  Hardee's  Taco John's  \
Month                                                               
February      517.0        130.0     228.0     124.0        211.0   
January       899.0        118.0     293.0     127.0        446.0   
Change       -382.0         12.0     -65.0      -3.0       -235.0   
% Change      -42.5         10.2     -22.2      -2.4        -52.7   

Brand     Wienerschnitzel        
Month                            
February            499.0  87.0  
January             858.0  48.0  
Change             -359.0  39.0  
% Change            -41.8  81.2  
